# 🟤 Bronze Layer – Data Ingestion (Auto Loader)

## 📌 Objective

Ingest raw data from ADLS (raw layer) into Delta tables using **Auto Loader**.

## ⚙️ Key Features

* Incremental file ingestion
* Schema inference & evolution
* Fault tolerance using checkpointing
* Scalable ingestion pipeline

## 📂 Source

* ADLS Raw Layer (`abfss://raw/...`)

## 📥 Target

* Bronze Delta Tables (`game_analytics.bronze.*`)

## 🧠 Concepts Used

* Auto Loader (`cloudFiles`)
* Schema Location
* Checkpointing
* Incremental Processing

## 🔄 Workflow

1. Select entity using widget
2. Read data using Auto Loader
3. Add ingestion timestamp
4. Write to Delta table

## 📌 Notes

* No transformations applied
* Raw data preserved as-is

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
%sql
USE CATALOG game_analytics;
USE SCHEMA bronze;

In [0]:
dbutils.widgets.dropdown(
    name='entity_name',
    defaultValue='players',
    choices=["players", "species", "maps", "equipments/rods", "equipments/baits", "equipments/rod_capability", "equipments/bait_capability", "events"],
    label="Entity Name"
)

In [0]:
raw_path = 'abfss://raw@storageaccgameanalytics.dfs.core.windows.net'
bronze_path = 'abfss://curated@storageaccgameanalytics.dfs.core.windows.net/bronze'

entity_name = dbutils.widgets.get('entity_name')

entity_clean = entity_name.split('/')[-1]
table_path = f'game_analytics.bronze.{entity_clean}'

source_path = f'{raw_path}/{entity_name}'
checkpoint_path = f'{bronze_path}/{entity_name}/checkpoint'
schema_path = f'{bronze_path}/{entity_name}/schema'
data_path = f'{bronze_path}/{entity_name}/data'

df = spark.readStream\
    .format('cloudFiles')\
    .option('cloudFiles.format', 'csv')\
    .option('cloudFiles.schemaLocation', schema_path)\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")\
    .option('header', 'true')\
    .option('inferSchema', 'true')\
    .load(source_path)

df = df.withColumn('ingestion_time', current_timestamp())

df.writeStream\
    .format('delta')\
    .outputMode('append')\
    .option('checkpointLocation',checkpoint_path)\
    .option('path', data_path)\
    .trigger(availableNow=True)\
    .toTable(table_path)